In [32]:
import os
import json
import math
import re
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from peft import PeftModel
from transformers import AutoTokenizer, AutoConfig, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [33]:
!pip install streamlit

In [34]:
MODEL_DIR = "/content/B7_light_best_model"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [35]:
with open(os.path.join(MODEL_DIR, "export_meta.json"), "r", encoding="utf-8") as f:
    META = json.load(f)

MODEL_NAME = META["model_name"]
MAX_LENGTH = META["max_length"]
HEAD_DROPOUT = META["head_dropout"]

TARGET_COLS = META["target_cols"]

TR_FEATURE_COLS = META["tr_feature_cols"]
CC_FEATURE_COLS = META["cc_feature_cols"]
LR_FEATURE_COLS = META["lr_feature_cols"]
GRA_FEATURE_COLS = META["gra_feature_cols"]

tr_feat_mean = np.array(META["tr_feat_mean"], dtype=np.float32)
tr_feat_std = np.array(META["tr_feat_std"], dtype=np.float32)

cc_feat_mean = np.array(META["cc_feat_mean"], dtype=np.float32)
cc_feat_std = np.array(META["cc_feat_std"], dtype=np.float32)

lr_feat_mean = np.array(META["lr_feat_mean"], dtype=np.float32)
lr_feat_std = np.array(META["lr_feat_std"], dtype=np.float32)

gra_feat_mean = np.array(META["gra_feat_mean"], dtype=np.float32)
gra_feat_std = np.array(META["gra_feat_std"], dtype=np.float32)

In [36]:
STOPWORDS = {
    "a","an","the","and","or","but","if","while","is","am","are","was","were",
    "be","been","being","of","to","in","on","for","with","as","at","by","from",
    "that","this","these","those","it","its","he","she","they","them","their",
    "we","our","you","your","i","me","my","mine","his","her","hers","do","does",
    "did","have","has","had","will","would","can","could","should","may","might",
    "not","so","than","then","there","here","about","into","over","after","before",
    "more","most","some","any","such","no","nor","too","very"
}

DISCOURSE_MARKERS = [
    "however", "therefore", "moreover", "furthermore", "in addition",
    "for example", "for instance", "on the one hand", "on the other hand",
    "in conclusion", "to conclude", "in summary", "as a result",
    "firstly", "secondly", "finally", "besides", "nevertheless",
    "thus", "overall", "in contrast", "for this reason"
]

LONG_WORD_MIN_LEN = 7
SCORE_MIN = 4.0
SCORE_MAX = 9.0


def safe_text(x):
    if x is None:
        return ""
    if isinstance(x, float) and np.isnan(x):
        return ""
    return str(x).strip()


def normalize_text(text):
    text = safe_text(text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize_words(text):
    text = normalize_text(text)
    return re.findall(r"[a-zA-Z']+", text)


def split_sentences(text):
    text = safe_text(text).strip()
    if not text:
        return []
    sents = re.split(r"(?<=[.!?])\s+", text)
    sents = [s.strip() for s in sents if s.strip()]
    return sents


def split_paragraphs(text):
    text = safe_text(text)
    paras = [p.strip() for p in re.split(r"\n\s*\n+", text) if p.strip()]
    if len(paras) == 0 and text.strip():
        paras = [text.strip()]
    return paras


def word_count(text):
    return len(tokenize_words(text))


def unique_ratio(words):
    if len(words) == 0:
        return 0.0
    return len(set(words)) / len(words)


def root_ttr(words):
    if len(words) == 0:
        return 0.0
    return len(set(words)) / math.sqrt(len(words))


def repetition_ratio(words):
    if len(words) <= 1:
        return 0.0
    c = Counter(words)
    repeated = sum(v for v in c.values() if v > 1)
    return repeated / len(words)


def repeated_word_ratio(words):
    if len(words) <= 1:
        return 0.0
    repeated_pairs = 0
    for i in range(1, len(words)):
        if words[i] == words[i - 1]:
            repeated_pairs += 1
    return repeated_pairs / len(words)


def avg_word_len(words):
    if len(words) == 0:
        return 0.0
    return float(np.mean([len(w) for w in words]))


def lexical_density_proxy(words):
    if len(words) == 0:
        return 0.0
    content_like = [w for w in words if len(w) > 3 and w not in STOPWORDS]
    return len(content_like) / len(words)


def long_word_ratio(words, min_len=LONG_WORD_MIN_LEN):
    if len(words) == 0:
        return 0.0
    return sum(len(w) >= min_len for w in words) / len(words)


def sentence_lengths(sentences):
    return [len(tokenize_words(s)) for s in sentences if len(tokenize_words(s)) > 0]


def count_discourse_markers(text):
    low = normalize_text(text)
    found = 0
    found_types = 0
    for m in DISCOURSE_MARKERS:
        c = low.count(m)
        if c > 0:
            found += c
            found_types += 1
    return found, found_types


def prompt_keywords(prompt):
    words = tokenize_words(prompt)
    words = [w for w in words if w not in STOPWORDS and len(w) > 2]
    return set(words)


def jaccard_coverage(prompt, essay):
    pk = prompt_keywords(prompt)
    ew = set(tokenize_words(essay))
    if len(pk) == 0:
        return 0.0
    return len(pk & ew) / len(pk)


def prompt_essay_similarity(prompt, essay):
    pw = prompt_keywords(prompt)
    ew = set([w for w in tokenize_words(essay) if w not in STOPWORDS and len(w) > 2])
    if len(pw) == 0 or len(ew) == 0:
        return 0.0
    return len(pw & ew) / math.sqrt(len(pw) * len(ew))


def has_opinion_statement(text):
    low = normalize_text(text)
    patterns = [
        "i believe", "i think", "in my opinion", "personally",
        "from my perspective", "it seems to me", "i would argue"
    ]
    return float(any(p in low for p in patterns))


def has_both_views(text):
    low = normalize_text(text)
    left = any(p in low for p in ["on the one hand", "some people think", "some people believe"])
    right = any(p in low for p in ["on the other hand", "others think", "others believe", "however"])
    return float(left and right)


def has_example(text):
    low = normalize_text(text)
    patterns = ["for example", "for instance", "such as"]
    return float(any(p in low for p in patterns))


def has_conclusion(text):
    low = normalize_text(text)
    patterns = ["in conclusion", "to conclude", "to sum up", "overall", "in summary"]
    return float(any(p in low for p in patterns))


def repeated_punct_ratio(text):
    if not text:
        return 0.0
    repeated = re.findall(r"([!?.,;:])\1+", text)
    return len(repeated) / max(len(text), 1)


def punct_density(text):
    if not text:
        return 0.0
    puncts = re.findall(r"[.!?,;:]", text)
    words = tokenize_words(text)
    return len(puncts) / max(len(words), 1)


def lowercase_sentence_start_ratio(text):
    sents = split_sentences(text)
    if len(sents) == 0:
        return 0.0
    bad = 0
    total = 0
    for s in sents:
        m = re.search(r"[A-Za-z]", s)
        if m:
            total += 1
            ch = s[m.start()]
            if ch.islower():
                bad += 1
    return bad / total if total > 0 else 0.0


def lowercase_i_ratio(text):
    tokens = re.findall(r"\b[iI]\b", safe_text(text))
    if len(tokens) == 0:
        return 0.0
    bad = sum(1 for t in tokens if t == "i")
    return bad / len(tokens)


def missing_terminal_punct(text):
    text = safe_text(text).strip()
    if not text:
        return 1.0
    return float(text[-1] not in ".!?")


def extract_tr_features(prompt, essay):
    return {
        "tr_prompt_essay_sim": float(prompt_essay_similarity(prompt, essay)),
        "tr_prompt_keyword_coverage": float(jaccard_coverage(prompt, essay)),
        "tr_has_opinion": float(has_opinion_statement(essay)),
        "tr_has_both_views": float(has_both_views(essay)),
        "tr_has_example": float(has_example(essay)),
        "tr_has_conclusion": float(has_conclusion(essay)),
        "tr_word_count": float(word_count(essay)),
    }


def extract_cc_features(prompt, essay):
    paras = split_paragraphs(essay)
    sents = split_sentences(essay)
    sent_lens = sentence_lengths(sents)
    dm_count, dm_div = count_discourse_markers(essay)

    para_lens = [word_count(p) for p in paras] if len(paras) > 0 else [0]

    return {
        "cc_num_paragraphs": float(len(paras)),
        "cc_avg_paragraph_len": float(np.mean(para_lens)) if len(para_lens) > 0 else 0.0,
        "cc_avg_sentence_len": float(np.mean(sent_lens)) if len(sent_lens) > 0 else 0.0,
        "cc_sentence_len_std": float(np.std(sent_lens)) if len(sent_lens) > 0 else 0.0,
        "cc_discourse_marker_count": float(dm_count),
        "cc_discourse_marker_diversity": float(dm_div),
    }


def extract_lr_features(prompt, essay):
    words = tokenize_words(essay)

    return {
        "lr_root_ttr": float(root_ttr(words)),
        "lr_avg_word_len": float(avg_word_len(words)),
        "lr_long_word_ratio": float(long_word_ratio(words)),
        "lr_repetition_ratio": float(repetition_ratio(words)),
        "lr_unique_word_ratio": float(unique_ratio(words)),
        "lr_lexical_density_proxy": float(lexical_density_proxy(words)),
    }


def extract_gra_features(prompt, essay):
    words = tokenize_words(essay)
    sents = split_sentences(essay)
    sent_lens = sentence_lengths(sents)

    return {
        "gf_word_count": float(len(words)),
        "gf_sentence_count": float(len(sents)),
        "gf_avg_sentence_len": float(np.mean(sent_lens)) if len(sent_lens) > 0 else 0.0,
        "gf_short_sentence_ratio": float(sum(l < 8 for l in sent_lens) / len(sent_lens)) if len(sent_lens) > 0 else 0.0,
        "gf_long_sentence_ratio": float(sum(l > 30 for l in sent_lens) / len(sent_lens)) if len(sent_lens) > 0 else 0.0,
        "gf_punct_density": float(punct_density(essay)),
        "gf_repeated_punct_ratio": float(repeated_punct_ratio(essay)),
        "gf_lowercase_sent_start_ratio": float(lowercase_sentence_start_ratio(essay)),
        "gf_lowercase_i_ratio": float(lowercase_i_ratio(essay)),
        "gf_repeated_word_ratio": float(repeated_word_ratio(words)),
        "gf_missing_terminal_punct": float(missing_terminal_punct(essay)),
    }


def build_input_text(prompt, essay):
    prompt = safe_text(prompt)
    essay = safe_text(essay)
    return (
        "You are an IELTS Writing examiner. "
        "Assess the essay based on four criteria: "
        "Task Response (TR), Coherence and Cohesion (CC), "
        "Lexical Resource (LR), and Grammatical Range and Accuracy (GRA).\n\n"
        "[PROMPT]\n"
        f"{prompt}\n\n"
        "[ESSAY]\n"
        f"{essay}"
    )


def standardize_features(feat_dict, cols, mean_, std_):
    arr = np.array([feat_dict[c] for c in cols], dtype=np.float32)
    std_ = np.where(std_ < 1e-6, 1.0, std_)
    arr = (arr - mean_) / std_
    return arr


def round_to_half(x):
    x = np.asarray(x, dtype=np.float32)
    return np.floor(x * 2 + 0.5) / 2

In [37]:
class QwenForIELTSMultiTask(nn.Module):
    def __init__(self, model_name: str, tokenizer, head_dropout: float = 0.1):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.pad_token_id = tokenizer.pad_token_id

        base_model = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        )
        base_model.config.pad_token_id = tokenizer.pad_token_id
        base_model.config.use_cache = False

        self.backbone = PeftModel.from_pretrained(base_model, MODEL_DIR)

        hidden_size = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(head_dropout)

        self.tr_feat_dim = len(TR_FEATURE_COLS)
        self.cc_feat_dim = len(CC_FEATURE_COLS)
        self.lr_feat_dim = len(LR_FEATURE_COLS)
        self.gra_feat_dim = len(GRA_FEATURE_COLS)

        self.tr_llm_head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(256, 1),
        )
        self.tr_feat_head = nn.Sequential(
            nn.Linear(self.tr_feat_dim, 64),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(64, 1),
        )
        self.tr_gate = nn.Sequential(
            nn.Linear(hidden_size + self.tr_feat_dim, 128),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(128, 1),
        )

        self.cc_llm_head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(256, 1),
        )
        self.cc_feat_head = nn.Sequential(
            nn.Linear(self.cc_feat_dim, 64),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(64, 1),
        )
        self.cc_gate = nn.Sequential(
            nn.Linear(hidden_size + self.cc_feat_dim, 128),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(128, 1),
        )

        self.lr_llm_head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(256, 1),
        )
        self.lr_feat_head = nn.Sequential(
            nn.Linear(self.lr_feat_dim, 64),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(64, 1),
        )
        self.lr_gate = nn.Sequential(
            nn.Linear(hidden_size + self.lr_feat_dim, 128),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(128, 1),
        )

        self.gra_llm_head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(256, 1),
        )
        self.gra_feat_head = nn.Sequential(
            nn.Linear(self.gra_feat_dim, 64),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(64, 1),
        )
        self.gra_gate = nn.Sequential(
            nn.Linear(hidden_size + self.gra_feat_dim, 128),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(128, 1),
        )

    def _last_token_pool(self, hidden_states, attention_mask):
        last_token_idx = attention_mask.sum(dim=1) - 1
        last_token_idx = last_token_idx.clamp(min=0)
        batch_idx = torch.arange(hidden_states.size(0), device=hidden_states.device)
        pooled = hidden_states[batch_idx, last_token_idx]
        return pooled

    def _fuse_score(self, pooled, feat, llm_head, feat_head, gate_net):
        llm_score = llm_head(pooled)
        feat_score = feat_head(feat)
        gate_input = torch.cat([pooled, feat], dim=1)
        gate = torch.sigmoid(gate_net(gate_input))
        fused_score = gate * llm_score + (1.0 - gate) * feat_score
        return fused_score, gate

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        tr_features=None,
        cc_features=None,
        lr_features=None,
        gra_features=None,
        **kwargs
    ):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs,
        )

        hidden_states = outputs.last_hidden_state
        pooled = self._last_token_pool(hidden_states, attention_mask)
        pooled = self.dropout(pooled)

        # ép pooled về dtype của custom heads để tránh lỗi bf16/float32
        head_dtype = self.tr_llm_head[0].weight.dtype
        pooled = pooled.to(dtype=head_dtype)

        tr_features = tr_features.to(device=pooled.device, dtype=head_dtype)
        cc_features = cc_features.to(device=pooled.device, dtype=head_dtype)
        lr_features = lr_features.to(device=pooled.device, dtype=head_dtype)
        gra_features = gra_features.to(device=pooled.device, dtype=head_dtype)

        tr, tr_gate = self._fuse_score(
            pooled, tr_features, self.tr_llm_head, self.tr_feat_head, self.tr_gate
        )
        cc, cc_gate = self._fuse_score(
            pooled, cc_features, self.cc_llm_head, self.cc_feat_head, self.cc_gate
        )
        lr, lr_gate = self._fuse_score(
            pooled, lr_features, self.lr_llm_head, self.lr_feat_head, self.lr_gate
        )
        gra, gra_gate = self._fuse_score(
            pooled, gra_features, self.gra_llm_head, self.gra_feat_head, self.gra_gate
        )

        combined_logits = torch.cat([tr, cc, lr, gra], dim=1)

        return {
            "logits": combined_logits,
            "tr_gate": tr_gate,
            "cc_gate": cc_gate,
            "lr_gate": lr_gate,
            "gra_gate": gra_gate,
        }

In [38]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = QwenForIELTSMultiTask(MODEL_NAME, tokenizer, head_dropout=HEAD_DROPOUT)

head_state = torch.load(
    os.path.join(MODEL_DIR, "custom_heads.pt"),
    map_location="cpu"
)

model.tr_llm_head.load_state_dict(head_state["tr_llm_head"])
model.tr_feat_head.load_state_dict(head_state["tr_feat_head"])
model.tr_gate.load_state_dict(head_state["tr_gate"])

model.cc_llm_head.load_state_dict(head_state["cc_llm_head"])
model.cc_feat_head.load_state_dict(head_state["cc_feat_head"])
model.cc_gate.load_state_dict(head_state["cc_gate"])

model.lr_llm_head.load_state_dict(head_state["lr_llm_head"])
model.lr_feat_head.load_state_dict(head_state["lr_feat_head"])
model.lr_gate.load_state_dict(head_state["lr_gate"])

model.gra_llm_head.load_state_dict(head_state["gra_llm_head"])
model.gra_feat_head.load_state_dict(head_state["gra_feat_head"])
model.gra_gate.load_state_dict(head_state["gra_gate"])

model.to(DEVICE)
model.eval()

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

QwenForIELTSMultiTask(
  (backbone): PeftModelForFeatureExtraction(
    (base_model): LoraModel(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj)

In [39]:
@torch.no_grad()
def predict_ielts(prompt: str, essay: str, round_band: bool = True):
    tr_feat = extract_tr_features(prompt, essay)
    cc_feat = extract_cc_features(prompt, essay)
    lr_feat = extract_lr_features(prompt, essay)
    gra_feat = extract_gra_features(prompt, essay)

    tr_arr = standardize_features(tr_feat, TR_FEATURE_COLS, tr_feat_mean, tr_feat_std)
    cc_arr = standardize_features(cc_feat, CC_FEATURE_COLS, cc_feat_mean, cc_feat_std)
    lr_arr = standardize_features(lr_feat, LR_FEATURE_COLS, lr_feat_mean, lr_feat_std)
    gra_arr = standardize_features(gra_feat, GRA_FEATURE_COLS, gra_feat_mean, gra_feat_std)

    text = build_input_text(prompt, essay)

    enc = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_tensors="pt"
    )

    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    tr_tensor = torch.tensor(tr_arr, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    cc_tensor = torch.tensor(cc_arr, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    lr_tensor = torch.tensor(lr_arr, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    gra_tensor = torch.tensor(gra_arr, dtype=torch.float32).unsqueeze(0).to(DEVICE)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        tr_features=tr_tensor,
        cc_features=cc_tensor,
        lr_features=lr_tensor,
        gra_features=gra_tensor,
    )

    preds = outputs["logits"].squeeze(0).detach().float().cpu().numpy()
    preds = np.clip(preds, SCORE_MIN, SCORE_MAX)

    raw_scores = {
        "TR": float(preds[0]),
        "CC": float(preds[1]),
        "LR": float(preds[2]),
        "GRA": float(preds[3]),
    }

    if round_band:
        final_scores = {k: float(round_to_half(v)) for k, v in raw_scores.items()}
    else:
        final_scores = raw_scores.copy()

    final_scores["Overall"] = float(round_to_half(np.mean(list(final_scores.values()))))

    gates = {
        "tr_gate": float(outputs["tr_gate"].squeeze().detach().float().cpu().item()),
        "cc_gate": float(outputs["cc_gate"].squeeze().detach().float().cpu().item()),
        "lr_gate": float(outputs["lr_gate"].squeeze().detach().float().cpu().item()),
        "gra_gate": float(outputs["gra_gate"].squeeze().detach().float().cpu().item()),
    }

    return {
        "raw_scores": raw_scores,
        "final_scores": final_scores,
        "gates": gates,
        "features": {
            "tr_features_raw": tr_feat,
            "cc_features_raw": cc_feat,
            "lr_features_raw": lr_feat,
            "gra_features_raw": gra_feat,
        }
    }

In [40]:
def predict_dataframe(df, prompt_col="prompt", essay_col="essay", round_band=True):
    rows = []
    for _, row in df.iterrows():
        result = predict_ielts(
            prompt=safe_text(row[prompt_col]),
            essay=safe_text(row[essay_col]),
            round_band=round_band
        )
        out = dict(row)
        out.update({
            "pred_TR": result["final_scores"]["TR"],
            "pred_CC": result["final_scores"]["CC"],
            "pred_LR": result["final_scores"]["LR"],
            "pred_GRA": result["final_scores"]["GRA"],
            "pred_Overall": result["final_scores"]["Overall"],

            "raw_TR": result["raw_scores"]["TR"],
            "raw_CC": result["raw_scores"]["CC"],
            "raw_LR": result["raw_scores"]["LR"],
            "raw_GRA": result["raw_scores"]["GRA"],

            "tr_gate": result["gates"]["tr_gate"],
            "cc_gate": result["gates"]["cc_gate"],
            "lr_gate": result["gates"]["lr_gate"],
            "gra_gate": result["gates"]["gra_gate"],
        })
        rows.append(out)
    return pd.DataFrame(rows)


In [41]:
if __name__ == "__main__":
    sample_prompt = """
The use of social media is replacing face to face interaction among many people in society. Do you think the advantages outweigh the disadvantages?
""".strip()

    sample_essay = """
Social media has deeply infiltrated into everyone’s life and is believed to be replacing our face-to-face interaction. This situation, although advantageous in certain aspects, is generally a detriment to true human communication in the long run.

On the one hand, there are a number of benefits from using social networks to communicate. Firstly, they facilitate communication in the modern times as now people can globally connect with old friends and relatives or with others who share common interests. For example, Facebook is currently providing service for 2.4 billion users who can choose to connect and interact with anyone they want, regardless of where they are. Secondly, study sessions are frequently happening on social networking websites through live streaming services. Therefore, learners around the world now can have free access to online classes on such sites.

On the other hand, the disadvantages of social media replacing face-to-face interaction are much more concerning. As these sites are becoming more and more dominant and attract large numbers of new users every day, people can fall prey to online communication abuse, such as online bullying and harassment. In fact, social networking websites can become a toxic environment where users can be verbally assaulted because there are only a few rules, most of which are spoken rules rather than established guidelines, that restrict hateful or abusive contents. Furthermore, overuse of social media to communicate can lead to people downplaying the importance of face-to-face interaction on which true human relationships thrive. Nowadays, many young and reclusive users prefer living in a virtual world on social sites than engaging in real-life relationships. This may have serious mental effects, such as increased stress, anxiety and loneliness.

In conclusion, the downsides of social media replacing face-to-face interaction are more significant than the benefits users could reap from those sites.
""".strip()

    result = predict_ielts(sample_prompt, sample_essay, round_band=True)

    print("=== FINAL SCORES ===")
    print(result["final_scores"])

    print("\n=== RAW SCORES ===")
    print(result["raw_scores"])

    print("\n=== GATES ===")
    print(result["gates"])

=== FINAL SCORES ===
{'TR': 7.5, 'CC': 7.5, 'LR': 7.5, 'GRA': 7.5, 'Overall': 7.5}

=== RAW SCORES ===
{'TR': 7.305397033691406, 'CC': 7.317463397979736, 'LR': 7.420095920562744, 'GRA': 7.387031555175781}

=== GATES ===
{'tr_gate': 0.999909520149231, 'cc_gate': 0.9998551607131958, 'lr_gate': 0.9998552799224854, 'gra_gate': 0.9998247027397156}


In [42]:
if __name__ == "__main__":
    sample_prompt = """
Environmental protection should be the responsibility of politicians, not individuals as individuals can do too little. To what extent do you agree or disagree?
""".strip()

    sample_essay = """
Some people think that politicians should be responsible for protecting the environment as there is nothing much that individuals can do about this problem. In my opinion, the responsibility to protect the environment should not fall upon politicians alone because ordinary citizens can make a significant contribution.

Firstly, politicians can urge the government to impose new laws against actions that damage the environment. For instance, one of the major factors leading to environmental pollution is the overuse of plastic products, like bottles and bags, and this can be stopped if the government issues an official ban on all companies from using plastic packaging. In addition, the rate of deforestation can also be reduced if high-ranking bureaucrats agree to impose strict punishments, such as long-term imprisonment and heavy fines, on those who cut down trees illegally. However, besides introducing and enforcing new laws and regulations, I doubt that there is any further action that politicians can take to protect the environment.

On the other hand, I believe that ordinary people, through small, everyday actions, can also greatly contribute to protecting the environment. First, citizens in many countries, like the Netherlands, have now shifted towards using bicycles and subway trains for their daily travel instead of cars, which has so far helped reduce a tremendous amount of CO2 released into the air, and improved air quality. Second, the problem of polluted oceans has also been tackled in many places thanks to groups of young people who voluntarily spend their time cleaning up beaches, or even diving into water to pick up trash. For example, many students in Nha Trang, a coastal city of Vietnam, spent nearly their whole summer holiday in 2018 keeping the beaches of their hometown clean by collecting all the trash from tourists, and even banned Chinese people from entering certain areas to prevent them from littering.

In conclusion, I hold the view that politicians alone cannot deal with all environmental problems, and therefore individuals should also make a contribution to protecting our environment.
""".strip()

    result = predict_ielts(sample_prompt, sample_essay, round_band=True)

    print("=== FINAL SCORES ===")
    print(result["final_scores"])

    print("\n=== RAW SCORES ===")
    print(result["raw_scores"])

    print("\n=== GATES ===")
    print(result["gates"])

=== FINAL SCORES ===
{'TR': 7.5, 'CC': 8.0, 'LR': 8.0, 'GRA': 7.5, 'Overall': 8.0}

=== RAW SCORES ===
{'TR': 7.696247100830078, 'CC': 7.768293857574463, 'LR': 7.87239933013916, 'GRA': 7.682231903076172}

=== GATES ===
{'tr_gate': 0.9999369382858276, 'cc_gate': 0.9998862743377686, 'lr_gate': 0.9998844861984253, 'gra_gate': 0.9999128580093384}


In [43]:
if __name__ == "__main__":
    sample_prompt = """
Leaders and directors in organizations are normally older people. Some people think having a younger leader would be better. To what extent do you agree or disagree?
""".strip()

    sample_essay = """
Society experiences a trend that the old in many organizations normally take over the leading and directing positions. Whereas, there is an opposite opinion to prefer to have a leader who is young. I totally agree with the idea that the people who lead and direct any organization should be the older generation. On the one hand, working with a person that becomes the header of a company at an early age would have many merits. First, they might put less pressure on the industriesâ€™ members. This is because they are not as strict as the older so it could create a more comfortable atmosphere to work. Second, these teenagers are more likely to be open-minded so it is rather easy to collaborate. Moreover, whenever receiving either positive or negative comments, they are always willing to accumulate them and adjust properly to have a more well-rounded perspective of working. On the other hand, though there are plenty of benefits brought by the younger headers, it is unacceptable to underestimate the abilities of mature leaders as well as directors. All of them (Timeâ€™s up) have a certain wealth of experiences after overcoming a wide range of challenges, which helps them to acquire the flexibility in tackling tough problems. In addition, they work with their high responsibilities and well-manner personalities, especially in Japan. For instance, Japanese industriesâ€™ CEOs pay much attention to the honesty of workers and penalize those who avoid their own work. In conclusion, despite the fact that it is quite good to have a young person run a company, I still believe that the better controller should be the older people. This trend is easy to understand due to the experiences and the personal traits of the old outweigh the less pressure and opened-mindedness of the younger CEOs.
""".strip()

    result = predict_ielts(sample_prompt, sample_essay, round_band=True)

    print("=== FINAL SCORES ===")
    print(result["final_scores"])

    print("\n=== RAW SCORES ===")
    print(result["raw_scores"])

    print("\n=== GATES ===")
    print(result["gates"])

=== FINAL SCORES ===
{'TR': 7.5, 'CC': 7.0, 'LR': 7.5, 'GRA': 7.0, 'Overall': 7.5}

=== RAW SCORES ===
{'TR': 7.3084611892700195, 'CC': 7.207231521606445, 'LR': 7.545064449310303, 'GRA': 7.226433753967285}

=== GATES ===
{'tr_gate': 0.9999191761016846, 'cc_gate': 0.9998342990875244, 'lr_gate': 0.9998376369476318, 'gra_gate': 0.9998328685760498}


In [44]:
if __name__ == "__main__":
    sample_prompt = """
Some people find advertisements amusing or annoying and they are not influenced by this when they shop. To what extend do you agree or disagree?
""".strip()

    sample_essay = """
these days that such advertisements can be is further important to the people. nowadays there is debate as to whether this had most positive or negative effects. in my opinion l strongly disagree that, that few number of individuals believe the advertisement is attractive or annoying sever for reasons On the one hand, some people believe that the advertisements amusing or annoying. Firstly ,the waste of time and money for example some companies we present products in advertisements we side it is more beneficial after they peopleâ€™s buy this product suddenly discovering the material is determined in addition waste of the time some individuals, such as reading book or doing sport but the advertisements is more positive of the society secondly, that such advertisements can be effective to the younger generation for instance, sometime the Tv we can see advertisement about new things after that the teenagers fast we sop this but they donâ€™t see the information about this products . On that one hand, l believes there are more benefits from the advertisement there a reactive the customers because the company they wonâ€™t customers also suggest the product for example in advertisements give easy information about materiel, price and quality furthermore presenting the products. In conclusion, it cannot be denied that advertisements play an important role in people's lives in my opinion l thank that such advertisements effective positive or negative.
""".strip()

    result = predict_ielts(sample_prompt, sample_essay, round_band=True)

    print("=== FINAL SCORES ===")
    print(result["final_scores"])

    print("\n=== RAW SCORES ===")
    print(result["raw_scores"])

    print("\n=== GATES ===")
    print(result["gates"])

=== FINAL SCORES ===
{'TR': 5.0, 'CC': 4.5, 'LR': 5.0, 'GRA': 4.5, 'Overall': 5.0}

=== RAW SCORES ===
{'TR': 4.805736064910889, 'CC': 4.532895088195801, 'LR': 5.04132080078125, 'GRA': 4.716699123382568}

=== GATES ===
{'tr_gate': 0.9997934699058533, 'cc_gate': 0.9996609687805176, 'lr_gate': 0.9995729327201843, 'gra_gate': 0.9995368719100952}


In [45]:
if __name__ == "__main__":
    sample_prompt = """
The increase in the production of consumer goods results in damage to the natural environment. What are the causes of this? What can be done to solve this problem?
""".strip()

    sample_essay = """
The pictures illustrate the structures of a city hospital in terms of 2007 and 2010. At a general glance, the bus stops were grouped together and relocated while two roundabouts on hospital Road were added. An entrance in the East of the hospital was added, leading to the newly built public parking lot and the space that the multipurpose car park originally located was then used for the staff-only. In 2007, the Ring Road only led to Hospital Road where the six bus stops and multipurpose parking lot were located on the two sides, after the reconstruction, besides the entrance to the employeesâ€™ car park, there was another one on the left side leading to the patientsâ€™ parking lot in 2010. Before 2010, the Ring Road was simply constructed around the City Hospital and led to the Hospital and City Road by only two paths, after some fixing periods, the paths going to the two roads in the South were generated into two roundabouts. Those new features additionally made way to the bus station which the bus stops grouped together after 2007.
""".strip()

    result = predict_ielts(sample_prompt, sample_essay, round_band=True)

    print("=== FINAL SCORES ===")
    print(result["final_scores"])

    print("\n=== RAW SCORES ===")
    print(result["raw_scores"])

    print("\n=== GATES ===")
    print(result["gates"])

=== FINAL SCORES ===
{'TR': 4.5, 'CC': 4.5, 'LR': 4.5, 'GRA': 4.5, 'Overall': 4.5}

=== RAW SCORES ===
{'TR': 4.291656970977783, 'CC': 4.31689453125, 'LR': 4.65904426574707, 'GRA': 4.381048202514648}

=== GATES ===
{'tr_gate': 0.9997275471687317, 'cc_gate': 0.9995481371879578, 'lr_gate': 0.999365508556366, 'gra_gate': 0.9982020854949951}


In [46]:
if __name__ == "__main__":
    sample_prompt = """
People nowadays sleep less than they used to in the past. What do you think is the reason for this? What are the effects of this habit?
""".strip()

    sample_essay = """
Nowadays, it is common that people do not spend long hours sleeping like they did in the past. This tendency is the result of several factors, which could lead to many impacts.

This phenomenon results from a host of different factors. One of them is that they have tight schedules and face pressure from study or work, while daytime is not sufficient for them to handle, which leads to them spending nighttime doing it. For example, there are increasingly high requirements for each subject that are incorporated into the curriculum such as presentations, teamwork assignments or individual homework in the Vietnamese education system. However, attending classes already takes up most of the day, the necessity of passing exams motivates students to sacrifice sleeping time in order to meet deadlines. Another important factor is the overuse of electronic devices. This is because people are addicted to many forms of entertainment on their electronic devices due to the rapid development of technology, which encourages them to stay awake late at night. As a result, bedtime is delayed, and sleep duration is significantly reduced.

Several related problems could result from this tendency. Firstly, lack of sleep can lead to serious effects for both physical and mental health. People who suffer from sleep deprivation can experience fatigue, weakened immunity, and make them undergo chronic pain related to mental and brain health problems such as anxiety and depression. Furthermore, this process in the long run can prevent productivity from efficiency, poor concentration, and thus they can not achieve goals in academic fields as well as future career paths, especially increasing the likelihood of accidents in some circumstances.

As a mentioned above, there are several factors causing less sleeping time and this could lead to several problems relating health and work productivity


""".strip()

    result = predict_ielts(sample_prompt, sample_essay, round_band=True)

    print("=== FINAL SCORES ===")
    print(result["final_scores"])

    print("\n=== RAW SCORES ===")
    print(result["raw_scores"])

    print("\n=== GATES ===")
    print(result["gates"])

=== FINAL SCORES ===
{'TR': 6.5, 'CC': 6.5, 'LR': 7.0, 'GRA': 6.5, 'Overall': 6.5}

=== RAW SCORES ===
{'TR': 6.536041259765625, 'CC': 6.728459358215332, 'LR': 7.178110122680664, 'GRA': 6.630632400512695}

=== GATES ===
{'tr_gate': 0.9999510049819946, 'cc_gate': 0.9999054670333862, 'lr_gate': 0.9999462366104126, 'gra_gate': 0.9999029636383057}


In [47]:
if __name__ == "__main__":
    sample_prompt = """
Employers should give longer holidays to employees to encourage them to do their job well. To what extent do you agree or disagree?
""".strip()

    sample_essay = """
Opinions are divided on whether giving longer holidays to employees is an effective way to motivate them to perform well at work. Although this choice brings about many advantages, its drawbacks far outweigh them.
Admittedly, this way of encouraging employees could bring certain benefits for many reasons. One of them is that doing so shows the care of employers for their employees, which creates a positive impression among staff. This is because longer holidays can provide them more time to relieve stress after long hours suffering from intensive pressure and excessive workload, thereby driving them to exert more effort to contribute their abilities for the company and improving the quality of work. Moreover, this action can reduce salary expenses during non-working periods, thus easing financial burden for them, especially small companies, and allocating funds to other operational needs.
Despite some aforementioned advantages, the disadvantages of this are more significant in several ways. Firstly, the worst part about this type of holiday is that it leaves people with a lasting vacation mood. This is due to this trend disruptive to people’s professional plans because they tend to put everything on hold for a short time, which results in decreasing productivity and motivation to work afterwards. In addition, the employer who launches this policy can face more challenges in a competitive market. This is because it is not a preferred trend for all employees, especially some workers who want to do more to get well-paid jobs. Thereby, the company is facing problems and damage to their reputation due to high employee turnover.
In conclusion, it is clear that drawbacks from giving longer holidays for employees outweigh benefits brought from this. Based on the presented arguments, I completely disagree with the proposed idea.

""".strip()

    result = predict_ielts(sample_prompt, sample_essay, round_band=True)

    print("=== FINAL SCORES ===")
    print(result["final_scores"])

    print("\n=== RAW SCORES ===")
    print(result["raw_scores"])

    print("\n=== GATES ===")
    print(result["gates"])

=== FINAL SCORES ===
{'TR': 6.5, 'CC': 6.5, 'LR': 6.5, 'GRA': 6.5, 'Overall': 6.5}

=== RAW SCORES ===
{'TR': 6.567577362060547, 'CC': 6.589033603668213, 'LR': 6.6882004737854, 'GRA': 6.617615222930908}

=== GATES ===
{'tr_gate': 0.9998769760131836, 'cc_gate': 0.9997896552085876, 'lr_gate': 0.9998036026954651, 'gra_gate': 0.9998396635055542}


In [48]:
if __name__ == "__main__":
    sample_prompt = """
Increase the price of petrol is the best way to solve growing traffic and pollution problems. To what extent do you agree or disagree?
""".strip()

    sample_essay = """
It is said that the increasement in gas prices is considered as the most optimal method in dealing with traffic congestion and pollution crisis. While this hypothesis may result in a positive impact on reversing the adversity, it may oppose some detrimental quality of life towards the people. In this essay, I would like to share my point of views on both sides of this problem.
It is true that in metropolis areas, traffic congestion has always been a hot topic and a pain-stakingly headache problem that challenges the authority on seeking out favorable solutions on how to resolve this. As said in the proposed method, increasing petrol prices can lead to people who own cars and motorbikes to use more environmentally friendly means of transport such as buses or bicycles. Another one to add is that it can alleviate the polluted environment with the reason being that less gas is exposed from cars and motorbikes. Lastly, because of the diminishing amount of car/motorbike drivers, it will resolve the immensely popullated petrol-exposed vehicles and therefore roads will become more broadened.
On the other hand, by doing so it might trigger dwellers’ behavioural perspectives towards this proposed solution. Firstly, for deprived families with cars or motorbikes as their main vehicles for commuting to work or school, how are they going to pay petrol gas with limited funds? What if there are other potential unfavorable conditions that they would not prefer to travel by buses or afford to buy a bicycle as such? Secondly, for safety issues, yes cars do have protection equipment if road accidents ever occurred and I can see why in most countries, people tend to use cars more than bicycles and buses so that if this proposed method were to be applied meaning the authority might put them in danger. Lastly, there are more variables and cases that affect the environment, not just the traffic so it is better to consider other possibilities in resolving this.
With all that being said, I do have mixed views between the right and wrong towards this statement. One being that this is a good way to encourage people to start using a more environmentally friendly means of transport for commuting. And the other one being that it is not entirely viable to other unfortunate families or main car/motorbike drivers on their basic needs.

""".strip()

    result = predict_ielts(sample_prompt, sample_essay, round_band=True)

    print("=== FINAL SCORES ===")
    print(result["final_scores"])

    print("\n=== RAW SCORES ===")
    print(result["raw_scores"])

    print("\n=== GATES ===")
    print(result["gates"])

=== FINAL SCORES ===
{'TR': 6.0, 'CC': 6.0, 'LR': 6.5, 'GRA': 6.0, 'Overall': 6.0}

=== RAW SCORES ===
{'TR': 6.217010498046875, 'CC': 6.194311618804932, 'LR': 6.4455084800720215, 'GRA': 6.108720779418945}

=== GATES ===
{'tr_gate': 0.9999167919158936, 'cc_gate': 0.9998711347579956, 'lr_gate': 0.9998289346694946, 'gra_gate': 0.9997778534889221}


In [49]:
if __name__ == "__main__":
    sample_prompt = """
International travel is becoming cheaper and more countries are opening their doors to tourists.
Do the advantages of this trend outweigh the disadvantages?
""".strip()

    sample_essay = """
It is clear that international travelling is becoming more affordable, and more nations are opening their borders to welcome tourists. This phenomenon gives lots of benefits to the host nation, but also leaves some problems and negative consequences to the development, economy, political and culture it owns.
On the one hand, tourism growth brings significant cultural benefits as well as economic benefits to the host countries. This greatly results in great revenue for the services sector, accommodation and transportation. Tourism growth also made career opportunities for laborers working in the tourist - attraction area, especially areas located in poor condition. Take Thailand and Vietnam as an example, we can witness that these two SEA nations are rapidly changing their economical structure due to tourism growth. Secondly, tourism growth also promotes cultural exchange. This is because local people can meet the opportunities to learn foreign languages, this indirectly improves the workers’ skill, and also helps to enhance cultural connections. For example, Japanese, Chinese and Korean have increased their needs for local guides who can speak their language fluently in Vietnam, giving great incomes and new opportunities for local people. Thirdly, the government has to upgrade their infrastructure such as: airports, highways, public transport. This not only provides and improves convenience for tourists, but also giving chances for local people to enhance their quality of life, indirectly helping other sectors of the economic structure to develop besides tourism.
Despite its clear benefits, mass tourism also creates several social and environmental issues. Firstly, too many tourists may lead to heavy amounts of wastes that would be harmful to the environment, and can be destructive to the ecosystem as well. Too many visitors also requires great demands of transport, which can lead to more CO2 exhaust to the environment. For example, Hanoi and HaLong Bay often meet trouble in waste management that greatly damages its air and water resources. Secondly, some nations that are heavily dependent on tourism can face an economic crisis. This is because tourism accounts for a significant role in its economic structure. For example, during COVID 19, the economy of ThaiLand has been dropped 2-3% in its GDP index due to the pandemic.
In conclusion, tourist growth has its pros and cons. I can firmly state that with proper management and sustainable policies, the advantages can outweigh the disadvantages.

""".strip()

    result = predict_ielts(sample_prompt, sample_essay, round_band=True)

    print("=== FINAL SCORES ===")
    print(result["final_scores"])

    print("\n=== RAW SCORES ===")
    print(result["raw_scores"])

    print("\n=== GATES ===")
    print(result["gates"])

=== FINAL SCORES ===
{'TR': 6.0, 'CC': 6.0, 'LR': 6.0, 'GRA': 6.0, 'Overall': 6.0}

=== RAW SCORES ===
{'TR': 5.8401079177856445, 'CC': 5.927818775177002, 'LR': 6.026633262634277, 'GRA': 5.9922661781311035}

=== GATES ===
{'tr_gate': 0.9998879432678223, 'cc_gate': 0.9997699856758118, 'lr_gate': 0.9997738003730774, 'gra_gate': 0.9998617172241211}


In [50]:
# -----------------------------
# 1. Embedding model
# -----------------------------
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_texts(texts, batch_size=32):
    return embedding_model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [51]:
# -----------------------------
# 2. Build retrieval texts
# -----------------------------
def build_doc_text(row):
    # Keep doc text simple for semantic retrieval
    return f"""
[PROMPT]
{safe_text(row['prompt'])}

[ESSAY]
{safe_text(row['essay'])}
""".strip()

def build_query_text(prompt, essay):
    return f"""
[PROMPT]
{safe_text(prompt)}

[ESSAY]
{safe_text(essay)}
""".strip()



In [52]:
# -----------------------------
# 3. Lightweight retrieval features
# -----------------------------
def extract_retrieval_features(prompt, essay):
    text = safe_text(essay)
    prompt_text = safe_text(prompt)

    words = re.findall(r"\b\w+\b", text.lower())
    prompt_words = re.findall(r"\b\w+\b", prompt_text.lower())

    sentences = re.split(r"[.!?]+", text)
    sentences = [s.strip() for s in sentences if s.strip()]

    essay_len = len(words)
    unique_words = len(set(words))
    lexical_diversity = unique_words / max(essay_len, 1)

    avg_sent_len = essay_len / max(len(sentences), 1)

    # Try to reuse your notebook TR features if available
    prompt_relevance = 0.0
    try:
        tr_feat = extract_tr_features(prompt, essay)
        if "prompt_relevance" in tr_feat:
            prompt_relevance = float(tr_feat["prompt_relevance"])
        else:
            # fallback: token overlap
            prompt_set = set([w for w in prompt_words if w not in STOPWORDS])
            essay_set = set([w for w in words if w not in STOPWORDS])
            if len(prompt_set) > 0:
                prompt_relevance = len(prompt_set & essay_set) / len(prompt_set)
    except Exception:
        prompt_set = set([w for w in prompt_words if w not in STOPWORDS])
        essay_set = set([w for w in words if w not in STOPWORDS])
        if len(prompt_set) > 0:
            prompt_relevance = len(prompt_set & essay_set) / len(prompt_set)

    # Simple readability proxy if you do not already have one
    # You can replace this later with your exact train-time feature
    readability_score = avg_sent_len

    return {
        "essay_len": float(essay_len),
        "prompt_relevance": float(prompt_relevance),
        "lexical_diversity": float(lexical_diversity),
        "readability_score": float(readability_score),
    }



In [53]:
# -----------------------------
# 4. Distance helpers
# -----------------------------
def normalize_diff(a, b, scale):
    return abs(float(a) - float(b)) / float(scale)

def quality_distance(row, pred_scores, feat_dict):
    score_dist = (
        normalize_diff(row["TR"], pred_scores["TR"], 9.0) +
        normalize_diff(row["CC"], pred_scores["CC"], 9.0) +
        normalize_diff(row["LR"], pred_scores["LR"], 9.0) +
        normalize_diff(row["GRA"], pred_scores["GRA"], 9.0)
    )

    feat_dist = (
        normalize_diff(row["essay_len"], feat_dict["essay_len"], 400.0) +
        normalize_diff(row["prompt_relevance"], feat_dict["prompt_relevance"], 1.0) +
        normalize_diff(row["lexical_diversity"], feat_dict["lexical_diversity"], 1.0) +
        normalize_diff(row["readability_score"], feat_dict["readability_score"], 100.0)
    )

    return score_dist + 0.5 * feat_dist



In [54]:
# -----------------------------
# 5. Retrieval core
# -----------------------------
def retrieve_cases(
    query_embedding,
    db_embeddings,
    df,
    pred_scores,
    feat_dict,
    top_k_vector=20,
    top_k_final=5,
):
    sims = cosine_similarity(query_embedding.reshape(1, -1), db_embeddings)[0]

    candidate_idx = np.argsort(-sims)[:top_k_vector]
    candidates = df.iloc[candidate_idx].copy()

    candidates["vector_sim"] = sims[candidate_idx]
    candidates["quality_dist"] = candidates.apply(
        lambda row: quality_distance(row, pred_scores, feat_dict),
        axis=1
    )

    # Hybrid score
    candidates["final_score"] = candidates["vector_sim"] - 0.7 * candidates["quality_dist"]

    # Optional: normalize version if you want more stable weighting
    # v = candidates["vector_sim"].values
    # q = candidates["quality_dist"].values
    # candidates["vector_sim_norm"] = (v - v.min()) / (v.max() - v.min() + 1e-8)
    # candidates["quality_dist_norm"] = (q - q.min()) / (q.max() - q.min() + 1e-8)
    # candidates["final_score"] = 0.6 * candidates["vector_sim_norm"] - 0.4 * candidates["quality_dist_norm"]

    candidates = candidates.sort_values("final_score", ascending=False)
    return candidates.head(top_k_final)



In [55]:
# -----------------------------
# 6. Build retrieval database
# -----------------------------
def build_retrieval_database(csv_path):
    df = pd.read_csv(csv_path)

    required_cols = [
        "prompt", "essay", "TR", "CC", "LR", "GRA", "band",
        "essay_len", "prompt_relevance", "lexical_diversity", "readability_score"
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in retrieval CSV: {missing}")

    if "evaluation" not in df.columns:
        df["evaluation"] = ""

    df["doc_text"] = df.apply(build_doc_text, axis=1)
    db_embeddings = embed_texts(df["doc_text"].tolist())

    return df, db_embeddings



In [56]:
# -----------------------------
# 7. Score model -> pred_scores
# -----------------------------
def get_pred_scores(prompt, essay):
    result = predict_ielts(prompt, essay, round_band=True)

    pred_scores = {
        "TR": float(result["final_scores"]["TR"]),
        "CC": float(result["final_scores"]["CC"]),
        "LR": float(result["final_scores"]["LR"]),
        "GRA": float(result["final_scores"]["GRA"]),
        "Overall": float(result["final_scores"]["Overall"]),
    }

    return result, pred_scores



In [57]:
# -----------------------------
# 8. Full pipeline for one essay
# -----------------------------
def retrieve_similar_essays_for_inference(
    prompt,
    essay,
    df,
    db_embeddings,
    top_k_vector=20,
    top_k_final=5,
):
    # 1. Predict scores
    pred_result, pred_scores = get_pred_scores(prompt, essay)

    # 2. Extract retrieval features
    feat_dict = extract_retrieval_features(prompt, essay)

    # 3. Build query embedding
    query_text = build_query_text(prompt, essay)
    query_embedding = embed_texts([query_text], batch_size=1)[0]

    # 4. Retrieve
    top_cases = retrieve_cases(
        query_embedding=query_embedding,
        db_embeddings=db_embeddings,
        df=df,
        pred_scores=pred_scores,
        feat_dict=feat_dict,
        top_k_vector=top_k_vector,
        top_k_final=top_k_final,
    )

    return {
        "pred_result": pred_result,
        "pred_scores": pred_scores,
        "feat_dict": feat_dict,
        "top_cases": top_cases,
    }



In [58]:
# -----------------------------
# 9. Pretty formatting
# -----------------------------
def format_retrieved_cases(top_cases, essay_char_limit=500):
    outputs = []

    for i, (_, row) in enumerate(top_cases.iterrows(), 1):
        essay_snippet = safe_text(row["essay"])[:essay_char_limit].replace("\n", " ").strip()
        evaluation_text = safe_text(row.get("evaluation", "")).strip()

        block = f"""
================ CASE {i} ================
Band: {row['band']}
TR={row['TR']} | CC={row['CC']} | LR={row['LR']} | GRA={row['GRA']}
vector_sim={row['vector_sim']:.4f} | quality_dist={row['quality_dist']:.4f} | final_score={row['final_score']:.4f}

Prompt:
{safe_text(row['prompt'])}

Essay snippet:
{essay_snippet}

Evaluation:
{evaluation_text[:400]}
""".strip()

        outputs.append(block)

    return "\n\n".join(outputs)



In [59]:
def split_evaluation_by_criteria(evaluation_text):
    sections = {
        "TR": "",
        "CC": "",
        "LR": "",
        "GRA": ""
    }

    text = safe_text(evaluation_text)
    if not text:
        return sections

    text = text.replace("\r", "\n")
    text = re.sub(r"\n{2,}", "\n", text).strip()

    patterns = [
        (r"(?:\*\*|##\s*)?\s*Task Achievement\s*:?", "TR"),
        (r"(?:\*\*|##\s*)?\s*Task Response\s*:?", "TR"),
        (r"(?:\*\*|##\s*)?\s*Coherence and Cohesion\s*:?", "CC"),
        (r"(?:\*\*|##\s*)?\s*Coherence\s*(?:and\s*Cohesion)?\s*:?", "CC"),
        (r"(?:\*\*|##\s*)?\s*Lexical Resource(?:\s*\(Vocabulary\))?\s*:?", "LR"),
        (r"(?:\*\*|##\s*)?\s*Vocabulary\s*:?", "LR"),
        (r"(?:\*\*|##\s*)?\s*Grammatical Range and Accuracy\s*:?", "GRA"),
        (r"(?:\*\*|##\s*)?\s*Grammar(?:\s*and\s*Accuracy)?\s*:?", "GRA"),
    ]

    matches = []
    for pat, crit in patterns:
        for m in re.finditer(pat, text, flags=re.I):
            matches.append((m.start(), m.end(), crit))

    if not matches:
        return sections

    seen = set()
    unique_matches = []
    for start, end, crit in sorted(matches, key=lambda x: x[0]):
        if crit not in seen:
            seen.add(crit)
            unique_matches.append((start, end, crit))

    for i, (_, end, crit) in enumerate(unique_matches):
        next_start = unique_matches[i + 1][0] if i + 1 < len(unique_matches) else len(text)
        body = text[end:next_start].strip()
        body = body.replace("**", " ").replace("##", " ")
        body = re.sub(r"\s+", " ", body).strip()
        sections[crit] = body

    return sections


def detect_prompt_type(prompt: str):
    p = normalize_text(prompt)

    if "advantages outweigh the disadvantages" in p or "advantages and disadvantages" in p:
        return "advantages_disadvantages"
    if "discuss both views" in p and "give your own opinion" in p:
        return "both_views_opinion"
    if "discuss both views" in p:
        return "both_views"
    if "to what extent do you agree or disagree" in p:
        return "agree_disagree"
    if "what are the causes" in p and "what can be done" in p:
        return "causes_solutions"
    if "what problems" in p and "what solutions" in p:
        return "problems_solutions"
    if "what are the reasons" in p and "what are the effects" in p:
        return "reasons_effects"
    return "other"


def filter_top_cases_same_prompt_type(prompt, top_cases, max_cases=5):
    if len(top_cases) == 0:
        return top_cases

    target_type = detect_prompt_type(prompt)
    rows = []

    for _, row in top_cases.iterrows():
        row_prompt = safe_text(row.get("prompt", ""))
        row_type = detect_prompt_type(row_prompt)
        if row_type == target_type:
            rows.append(row)

    if len(rows) == 0:
        return top_cases.head(max_cases)

    return pd.DataFrame(rows).head(max_cases)


def clean_reference_comment(text: str) -> str:
    text = safe_text(text)
    if not text:
        return ""

    text = text.replace("\r", " ").replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()

    text = re.sub(
        r"Suggested Band Score\s*(\([^)]*\))?\s*:\s*\*?\*?\d+(\.\d+)?\*?\*?",
        "",
        text,
        flags=re.I
    )
    text = re.sub(
        r"Overall Band Score\s*(\([^)]*\))?\s*:\s*\*?\*?\d+(\.\d+)?\*?\*?",
        "",
        text,
        flags=re.I
    )

    text = re.sub(r"^\d+(\.\d+)?\s*", "", text)
    text = re.sub(r"Feedback and Additional Comments:.*", "", text, flags=re.I)
    text = re.sub(r"Strengths:.*", "", text, flags=re.I)
    text = re.sub(r"Areas for Improvement:.*", "", text, flags=re.I)
    text = re.sub(r"Overall, the essay.*", "", text, flags=re.I)

    text = re.sub(r'For example,.*?(?=[.;]|$)', "", text, flags=re.I)
    text = re.sub(r'such as.*?(?=[.;]|$)', "", text, flags=re.I)
    text = re.sub(r'"[^"]+"\s+should\s+be\s+"[^"]+"', "", text, flags=re.I)

    text = text.replace("**", " ").replace("##", " ")
    text = re.sub(r"\s*[-*]\s*", " ", text)
    text = re.sub(r"\s{2,}", " ", text).strip(" -.;,")

    sents = re.split(r"(?<=[.!?])\s+", text)
    sents = [s.strip() for s in sents if s.strip()]
    text = " ".join(sents[:2]).strip()

    return text


def build_feedback_evidence(prompt, top_cases, max_cases=1, per_criterion=1):
    evidence = {
        "TR": [],
        "CC": [],
        "LR": [],
        "GRA": []
    }

    filtered_cases = filter_top_cases_same_prompt_type(
        prompt,
        top_cases,
        max_cases=max_cases
    )

    if len(filtered_cases) == 0:
        filtered_cases = top_cases.head(max_cases)

    if len(filtered_cases) == 0:
        return evidence

    row = filtered_cases.iloc[0]
    raw_eval = safe_text(row.get("evaluation", ""))
    split = split_evaluation_by_criteria(raw_eval)

    meta = {
        "band": row.get("band", ""),
        "TR_score": row.get("TR", ""),
        "CC_score": row.get("CC", ""),
        "LR_score": row.get("LR", ""),
        "GRA_score": row.get("GRA", ""),
        "vector_sim": row.get("vector_sim", ""),
        "quality_dist": row.get("quality_dist", ""),
        "final_score": row.get("final_score", ""),
    }

    for crit in ["TR", "CC", "LR", "GRA"]:
        raw_txt = split.get(crit, "")

        if not raw_txt:
            raw_txt = raw_eval

        clean_txt = clean_reference_comment(raw_txt)

        if clean_txt:
            evidence[crit].append({
                "meta": meta,
                "text": clean_txt
            })

    return evidence


def build_feedback_prompt(prompt, essay, pred_scores, top_cases):
    evidence = build_feedback_evidence(prompt, top_cases)

    blocks = []
    for crit in ["TR", "CC", "LR", "GRA"]:
        refs = evidence.get(crit, [])
        block_lines = [f"## {crit} references"]

        if len(refs) == 0:
            block_lines.append("No strong reference available.")
        else:
            for i, ref in enumerate(refs, 1):
                m = ref["meta"]
                t = ref["text"]
                block_lines.append(
                    f"[Ref {i}] band={m['band']}, "
                    f"TR={m['TR_score']}, CC={m['CC_score']}, LR={m['LR_score']}, GRA={m['GRA_score']}, "
                    f"sim={m['vector_sim']}, final={m['final_score']}\n"
                    f"{t}"
                )

        blocks.append("\n".join(block_lines))

    references_text = "\n\n".join(blocks)

    return f"""
You are an IELTS Writing Task 2 feedback writer.

Write concise, natural English feedback for four criteria:
TR, CC, LR, and GRA.

Use only these sources:
1. The current essay
2. The fixed predicted scores
3. The retrieved reference evaluations

Rules:
- Do not invent errors that are not supported by the essay or the references.
- Do not copy the reference comments verbatim.
- Do not output any score different from the fixed predicted scores.
- Keep the tone clear, useful, and teacher-like.
- For each criterion, include:
  1) one short strength,
  2) one or two score-limiting issues,
  3) one concrete revision suggestion.
- Mention comparison with stronger retrieved responses when it is justified by the references.
- Write in English only.

Return the result in exactly this format:

TR:
...

CC:
...

LR:
...

GRA:
...

[CURRENT PROMPT]
{prompt}

[CURRENT ESSAY]
{essay}

[FIXED PREDICTED SCORES]
TR={pred_scores['TR']}
CC={pred_scores['CC']}
LR={pred_scores['LR']}
GRA={pred_scores['GRA']}
Overall={pred_scores['Overall']}

[RETRIEVED REFERENCES]
{references_text}
""".strip()


def parse_feedback_output(text):
    result = {
        "TR": "",
        "CC": "",
        "LR": "",
        "GRA": ""
    }

    if not isinstance(text, str):
        return result

    matches = re.findall(
        r"(TR|CC|LR|GRA)\s*:\s*(.*?)(?=\n(?:TR|CC|LR|GRA)\s*:|\Z)",
        text,
        flags=re.S
    )

    for crit, body in matches:
        result[crit] = body.strip()

    return result


def generate_llm_feedback(prompt, essay, pred_scores, top_cases, writer_fn=None):
    """
    writer_fn: optional callable that takes prompt_text:str and returns generated text:str
    """

    prompt_text = build_feedback_prompt(
        prompt=prompt,
        essay=essay,
        pred_scores=pred_scores,
        top_cases=top_cases
    )

    if writer_fn is None:
        return {
            "TR": "Writer model/API not attached yet.",
            "CC": "Writer model/API not attached yet.",
            "LR": "Writer model/API not attached yet.",
            "GRA": "Writer model/API not attached yet.",
            "_prompt_preview": prompt_text
        }

    raw_output = writer_fn(prompt_text)
    parsed = parse_feedback_output(raw_output)

    if not any(parsed.values()):
        return {
            "TR": raw_output.strip(),
            "CC": "",
            "LR": "",
            "GRA": "",
            "_raw_output": raw_output
        }

    parsed["_raw_output"] = raw_output
    return parsed

In [60]:
TRAIN_CSV = "ielts_train_df.csv"

df_retrieval, db_embeddings = build_retrieval_database(TRAIN_CSV)

Batches:   0%|          | 0/207 [00:00<?, ?it/s]

In [61]:
def run_full_feedback_pipeline(
    prompt,
    essay,
    df,
    db_embeddings,
    writer_fn=None,
    top_k_vector=20,
    top_k_final=5,
):
    result = retrieve_similar_essays_for_inference(
        prompt=prompt,
        essay=essay,
        df=df,
        db_embeddings=db_embeddings,
        top_k_vector=top_k_vector,
        top_k_final=top_k_final,
    )

    feedback = generate_llm_feedback(
        prompt=prompt,
        essay=essay,
        pred_scores=result["pred_scores"],
        top_cases=result["top_cases"],
        writer_fn=writer_fn,
    )

    return {
        "pred_result": result["pred_result"],
        "pred_scores": result["pred_scores"],
        "feat_dict": result["feat_dict"],
        "top_cases": result["top_cases"],
        "feedback": feedback,
    }

In [62]:
sample_prompt = """
People nowadays sleep less than they used to in the past. What do you think is the reason for this? What are the effects of this habit?
""".strip()

sample_essay = """
Nowadays, it is common that people do not spend long hours sleeping like they did in the past. This tendency is the result of several factors, which could lead to many impacts.

This phenomenon results from a host of different factors. One of them is that they have tight schedules and face pressure from study or work, while daytime is not sufficient for them to handle, which leads to them spending nighttime doing it. For example, there are increasingly high requirements for each subject that are incorporated into the curriculum such as presentations, teamwork assignments or individual homework in the Vietnamese education system. However, attending classes already takes up most of the day, the necessity of passing exams motivates students to sacrifice sleeping time in order to meet deadlines. Another important factor is the overuse of electronic devices. This is because people are addicted to many forms of entertainment on their electronic devices due to the rapid development of technology, which encourages them to stay awake late at night. As a result, bedtime is delayed, and sleep duration is significantly reduced.

Several related problems could result from this tendency. Firstly, lack of sleep can lead to serious effects for both physical and mental health. People who suffer from sleep deprivation can experience fatigue, weakened immunity, and make them undergo chronic pain related to mental and brain health problems such as anxiety and depression. Furthermore, this process in the long run can prevent productivity from efficiency, poor concentration, and thus they can not achieve goals in academic fields as well as future career paths, especially increasing the likelihood of accidents in some circumstances.

As a mentioned above, there are several factors causing less sleeping time and this could lead to several problems relating health and work productivity
""".strip()

out = run_full_feedback_pipeline(
    prompt=sample_prompt,
    essay=sample_essay,
    df=df_retrieval,
    db_embeddings=db_embeddings,
    writer_fn=None,
)

print(out["pred_scores"])
print(format_retrieved_cases(out["top_cases"]))
print(out["feedback"].get("_prompt_preview", ""))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{'TR': 6.5, 'CC': 6.5, 'LR': 7.0, 'GRA': 6.5, 'Overall': 6.5}
================ CASE 1 ================
Band: 6.5
TR=6.5 | CC=6.5 | LR=6.5 | GRA=6.5
vector_sim=0.6715 | quality_dist=0.3818 | final_score=0.4042

Prompt:
Children find it difficult to concentrate on or pay attention to their studies in school. What are the reasons? How can we solve this problem?

Essay snippet:
Education is the key to becoming successful in life, however, most of the students find it difficult to concentrate on studying at school. Although they feel distracted, there are a lot of courses that students need to cover, and I will discuss possible solutions for improving their concentration in the upcoming paragraphs. Firstly, the main reason that the students are unable to attention to study is that they feel fear because of the extra syllabus as they feel they are unable to finish on tim

Evaluation:
**Task Achievement:**

- The essay effectively addresses the task prompt by discussing reasons for children's